<a href="https://colab.research.google.com/github/JeffersonRodrigues9/Automacoes_com_python/blob/main/Extra%C3%A7%C3%A3o_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import fitz  # PyMuPDF
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def normalizar_data(data_str):
    if not data_str:
        return ""
    return data_str.replace("-", "/")

import re

def extrair_valor_nominal_3(texto):
    bloco = re.search(
        r'Valor\s*Da\s*Parcela([\s\S]+?)(?=Data|Assinatura|$)',
        texto,
        flags=re.IGNORECASE
    )

    if not bloco:
        return ""

    valores = re.findall(r'R\$\s*([\d\.,]+)', bloco.group(1))

    if len(valores) >= 3:
        return valores[2]

    return ""

def extrair_cnpjs_endosso(texto):
    bloco = re.search(
        r'ENDOSSO\s*DIGITAL([\s\S]+)',
        texto,
        flags=re.IGNORECASE
    )

    if not bloco:
        return "", ""

    trecho = bloco.group(1)

    cnpjs = re.findall(
        r'\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b',
        trecho
    )

    cnpj1 = cnpjs[0] if len(cnpjs) > 0 else ""
    cnpj2 = cnpjs[1] if len(cnpjs) > 1 else ""

    return cnpj1, cnpj2


def extrair_dados_pdf(caminho_pdf):
    try:
        texto_completo = ""

        with fitz.open(caminho_pdf) as pdf:
            texto_completo = "\n".join(
                pagina.get_text("text") for pagina in pdf
            )


        def normalizar_data(data_str):
            if not data_str:
                return ""
            return data_str.replace("-", "/")

        cnpj_cedente = re.search(
            r'sob\s*\n?\s*o nº\.?\s*([\d\.\/\-]+)',
            texto_completo,
            flags=re.IGNORECASE
        )

        cpf_sacado = re.search(
            r'CPF[:\s]*([\d\.\-]+)',
            texto_completo,
            flags=re.IGNORECASE
        )

        numero_documento = re.search(
            r'CÉDULA DE CRÉDITO BANCÁRIO\s*n[°º]\s*([\w\-\/\.]+)',
            texto_completo,
            flags=re.IGNORECASE
        )

        valor_parcela = re.search(
            r'Valor\s*Da\s*Parcela[:\s]*R\$\s*([\d\.,]+)',
            texto_completo,
            flags=re.IGNORECASE
        )

        data_emissao = re.search(
            r'Data\s*de\s*Emissão[\s\S]*?(\d{2}[-/]\d{2}[-/]\d{4})',
            texto_completo,
            flags=re.IGNORECASE
        )

        data_vencimento = re.search(
            r'Data\s*de\s*Vencimento[\s\S]*?(\d{2}[-/]\d{2}[-/]\d{4})',
            texto_completo,
            flags=re.IGNORECASE
        )

        assinatura = re.search(
            r'Assinaturas\s*([\w\s\.ÁÉÍÓÚÂÊÔÃÕÇáéíóúâêôãõç]+)',
            texto_completo,
            flags=re.IGNORECASE
        )

        cnpj_endossante, cnpj_endossatario = extrair_cnpjs_endosso(texto_completo)

        return {
            "Arquivo": os.path.basename(caminho_pdf),
            "DOC_CEDENTE": cnpj_cedente.group(1) if cnpj_cedente else "",
            "DOC_SACADO": cpf_sacado.group(1) if cpf_sacado else "",
            "NU_DOCUMENTO_AJUSTADO": numero_documento.group(1) if numero_documento else "",
            "VALOR_NOMINAL": extrair_valor_nominal_3(texto_completo),
            "DATA_EMISSAO": normalizar_data(data_emissao.group(1)) if data_emissao else "",
            "DATA_VENCIMENTO": normalizar_data(data_vencimento.group(1)) if data_vencimento else "",
            "ASSINATURA": assinatura.group(1).strip() if assinatura else "",
            "CNPJ_ENDOSSANTE": cnpj_endossante,
            "CNPJ_ENDOSSATARIO": cnpj_endossatario
        }

    except Exception as e:
        return {
            "Arquivo": os.path.basename(caminho_pdf),
            "Erro": f"{os.path.basename(caminho_pdf)}: {str(e)}"
        }

def processar_pasta_pdfs(pasta_pdf, caminho_saida_excel, num_processos=8):
    arquivos_pdf = [
        os.path.join(pasta_pdf, f)
        for f in os.listdir(pasta_pdf)
        if f.lower().endswith(".pdf")
    ]
    resultados = []

    with ProcessPoolExecutor(max_workers=num_processos) as executor:
        futuros = {executor.submit(extrair_dados_pdf, caminho): caminho for caminho in arquivos_pdf}
        for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Processando PDFs"):
            resultados.append(futuro.result())

    df = pd.DataFrame(resultados)
    df.to_excel(caminho_saida_excel, index=False)
    print(f"\nArquivo Excel gerado em: {caminho_saida_excel}")

if __name__ == "__main__":
    pasta_pdf = r""
    saida_excel = r".xlsx"
    processar_pasta_pdfs(pasta_pdf, saida_excel, num_processos=12)